# Phase 6: Cold Start System
## FreshCart AI — 3-Tier Cold Start Routing

This notebook builds the cold start recommendation system for FreshCart AI:
- **Tier 1** (0 orders): Global popularity — top 20 products across all users
- **Tier 2** (1–2 orders): Aisle affinity — top products from user's most-purchased aisle
- **Tier 3** (3+ orders): LSTM+Time model — fully personalized sequential recommendations

The precomputed JSON tables (Tier 1 & 2) enable instant recommendations without model inference for cold start users.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

# --- Path configuration ---
# Running locally — adjust DATA_DIR if running on Colab:
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/freshcart/data/processed'
# PRECOMPUTED_DIR = '/content/drive/MyDrive/freshcart/data/precomputed'
# MODELS_DIR = '/content/drive/MyDrive/freshcart/saved_models'

DATA_DIR = '../data/processed'
PRECOMPUTED_DIR = '../data/precomputed'
MODELS_DIR = '../saved_models'

os.makedirs(PRECOMPUTED_DIR, exist_ok=True)
print('Directories ready.')

In [ ]:
# Load source data
orders = pd.read_csv(f'{DATA_DIR}/orders_subset.csv')
order_products = pd.read_csv(f'{DATA_DIR}/order_products_subset.csv')
products = pd.read_csv(f'{DATA_DIR}/products.csv')
aisles = pd.read_csv(f'{DATA_DIR}/aisles.csv')

print(f'Orders: {orders.shape}')
print(f'Order-Products: {order_products.shape}')
print(f'Products: {products.shape}')
print(f'Aisles: {aisles.shape}')

## Tier 1: Global Popularity (0 orders)

Tier 1 serves the 20 most globally popular products to brand-new users with no order history. This lookup table is static and computed once from the full dataset.

In [ ]:
# Count total times each product was ordered across all users
# Per D-01: store only product_id arrays (no full product objects)
product_counts = order_products['product_id'].value_counts()
global_top20 = [int(x) for x in product_counts.head(20).index.tolist()]

with open(f'{PRECOMPUTED_DIR}/global_top20.json', 'w') as f:
    json.dump(global_top20, f)

print(f'Global Top 20 product IDs: {global_top20}')
print(f'Saved to {PRECOMPUTED_DIR}/global_top20.json')

## Time-Context Lookup Tables

Hourly and day-of-week popularity tables power context-aware cold start recommendations. These tables capture temporal shopping patterns (e.g., coffee in the morning, wine on weekends) so even cold start users receive time-relevant suggestions.

In [ ]:
# Join orders with order_products to get time context for each purchase
op_with_time = order_products.merge(
    orders[['order_id', 'order_hour_of_day', 'order_dow']],
    on='order_id'
)

# Top 10 products per hour (0–23)
hourly_top10 = {}
for hour in range(24):
    hour_products = op_with_time[op_with_time['order_hour_of_day'] == hour]
    top10 = [int(x) for x in hour_products['product_id'].value_counts().head(10).index.tolist()]
    hourly_top10[str(hour)] = top10  # JSON keys must be strings

with open(f'{PRECOMPUTED_DIR}/hourly_top10.json', 'w') as f:
    json.dump(hourly_top10, f)

print(f'Hourly Top 10: {len(hourly_top10)} hours')
print(f'Sample (hour 10): {hourly_top10["10"]}')

In [ ]:
# Top 10 products per day of week (0=Sunday, 6=Saturday in Instacart)
dow_top10 = {}
for dow in range(7):
    dow_products = op_with_time[op_with_time['order_dow'] == dow]
    top10 = [int(x) for x in dow_products['product_id'].value_counts().head(10).index.tolist()]
    dow_top10[str(dow)] = top10

with open(f'{PRECOMPUTED_DIR}/dow_top10.json', 'w') as f:
    json.dump(dow_top10, f)

print(f'DOW Top 10: {len(dow_top10)} days')
print(f'Sample (day 0 = Sunday): {dow_top10["0"]}')

## Tier 2: Aisle-Affinity Lookup

Tier 2 users (1–2 orders) receive recommendations from their most-purchased aisle. This table precomputes the top 10 products per aisle across all 134 Instacart aisles, enabling instant aisle-affinity lookups without DataFrame joins at query time.

In [ ]:
# Join order_products with products to get aisle_id for each purchase
op_with_aisle = order_products.merge(
    products[['product_id', 'aisle_id']],
    on='product_id'
)

# Top 10 products per aisle (by purchase frequency)
aisle_top10 = {}
for aisle_id in sorted(op_with_aisle['aisle_id'].unique()):
    aisle_products = op_with_aisle[op_with_aisle['aisle_id'] == aisle_id]
    top10 = [int(x) for x in aisle_products['product_id'].value_counts().head(10).index.tolist()]
    aisle_top10[str(int(aisle_id))] = top10

with open(f'{PRECOMPUTED_DIR}/aisle_top10.json', 'w') as f:
    json.dump(aisle_top10, f)

print(f'Aisle Top 10: {len(aisle_top10)} aisles')
# Show a sample aisle with product names
sample_aisle_id = list(aisle_top10.keys())[0]
sample_pids = aisle_top10[sample_aisle_id]
sample_names = products[products['product_id'].isin(sample_pids)]['product_name'].tolist()
print(f'Sample aisle {sample_aisle_id}: {sample_names[:5]}')

## Precomputation Validation

Verify all 4 JSON files have the correct structure before proceeding to routing logic.

In [ ]:
print('=' * 60)
print('Precomputation Validation')
print('=' * 60)

with open(f'{PRECOMPUTED_DIR}/global_top20.json') as f:
    g = json.load(f)
assert isinstance(g, list) and len(g) == 20
assert all(isinstance(x, int) for x in g)
print(f'global_top20.json: {len(g)} product IDs [OK]')

with open(f'{PRECOMPUTED_DIR}/hourly_top10.json') as f:
    h = json.load(f)
assert len(h) == 24
assert all(str(i) in h for i in range(24))
assert all(len(v) == 10 for v in h.values())
print(f'hourly_top10.json: {len(h)} hours x 10 products [OK]')

with open(f'{PRECOMPUTED_DIR}/dow_top10.json') as f:
    d = json.load(f)
assert len(d) == 7
assert all(str(i) in d for i in range(7))
assert all(len(v) == 10 for v in d.values())
print(f'dow_top10.json: {len(d)} days x 10 products [OK]')

with open(f'{PRECOMPUTED_DIR}/aisle_top10.json') as f:
    a = json.load(f)
assert len(a) >= 10
assert all(len(v) <= 10 for v in a.values())
assert all(isinstance(pid, int) for v in a.values() for pid in v)
print(f'aisle_top10.json: {len(a)} aisles, up to 10 products each [OK]')

print('=' * 60)
print('All 4 precomputed JSON files validated successfully.')
print('=' * 60)

## Tier Routing Logic

The 3-tier cold start routing system:
- **Tier 1** (0 orders): Global popularity — no user history needed
- **Tier 2** (1–2 orders): Aisle affinity — uses the user's most-purchased aisle
- **Tier 3** (3+ orders): LSTM+Time model — full personalized recommendations

In [ ]:
def get_user_tier(order_count):
    """Determine recommendation tier based on user's order history.

    Tier 1: 0 orders (brand new user) -> global popularity
    Tier 2: 1-2 orders (minimal history) -> aisle affinity
    Tier 3: 3+ orders (sufficient history) -> LSTM+Time model
    """
    if order_count == 0:
        return 1
    elif order_count <= 2:
        return 2
    else:
        return 3

# Verify tier routing
assert get_user_tier(0) == 1, '0 orders should be Tier 1'
assert get_user_tier(1) == 2, '1 order should be Tier 2'
assert get_user_tier(2) == 2, '2 orders should be Tier 2'
assert get_user_tier(3) == 3, '3 orders should be Tier 3'
assert get_user_tier(6) == 3, '6 orders should be Tier 3'
assert get_user_tier(100) == 3, '100 orders should be Tier 3'
print('get_user_tier: all assertions passed')

In [ ]:
# Load precomputed lookup tables
with open(f'{PRECOMPUTED_DIR}/global_top20.json') as f:
    global_top20 = json.load(f)

with open(f'{PRECOMPUTED_DIR}/hourly_top10.json') as f:
    hourly_top10 = json.load(f)

with open(f'{PRECOMPUTED_DIR}/dow_top10.json') as f:
    dow_top10 = json.load(f)

with open(f'{PRECOMPUTED_DIR}/aisle_top10.json') as f:
    aisle_top10 = json.load(f)

# Load vocab for LSTM inference
with open(f'{DATA_DIR}/vocab.json') as f:
    vocab = json.load(f)

# Build reverse mapping: index -> product_id
index_to_product = {int(v): int(k) for k, v in vocab.items()}

# IMPORTANT: Precompute product_aisles as a global dict so recommend_tier2
# does NOT depend on a bare 'products' DataFrame. Ensures portability
# when imported outside notebook context in Phase 8 FastAPI backend.
product_aisles = {row['product_id']: row['aisle_id'] for _, row in products.iterrows()}

print(f'Loaded: global_top20 ({len(global_top20)}), hourly ({len(hourly_top10)}h), dow ({len(dow_top10)}d), aisle ({len(aisle_top10)} aisles)')
print(f'Vocab size: {len(vocab)}')
print(f'product_aisles: {len(product_aisles)} product->aisle mappings')

In [ ]:
def recommend_tier1(n=5):
    """Tier 1: Return top-N globally popular products.

    For brand-new users with 0 orders. Simply slices from global_top20.
    Returns: list of product_id integers, length = n
    """
    return global_top20[:n]

# Test
t1_recs = recommend_tier1(5)
print(f'Tier 1 recommendations (top 5): {t1_recs}')
assert len(t1_recs) == 5
assert all(isinstance(x, int) for x in t1_recs)
print('recommend_tier1: assertions passed')

In [ ]:
def recommend_tier2(user_order_history, cart_items=None, n=5):
    """Tier 2: Aisle-affinity recommendations for users with 1-2 orders.

    Args:
        user_order_history: list of product_ids from user's past orders
        cart_items: list of product_ids currently in cart (filter out per D-02)
        n: number of recommendations to return (default 5)

    Returns: list of exactly n product_id integers

    Logic (per context decisions):
        D-01: Uses precomputed aisle_top10 (product_id arrays only)
        D-02: Filters cart items; past purchases allowed
        D-03: Pads with global_top20 if < n results

    Note: Uses precomputed product_aisles dict (not bare products DataFrame)
    to ensure portability outside notebook context.
    """
    if cart_items is None:
        cart_items = []
    cart_set = set(cart_items)

    # Find user's top aisle by counting product purchases per aisle
    aisle_counts = {}
    for pid in user_order_history:
        aisle_id = product_aisles.get(pid)
        if aisle_id is not None:
            aisle_counts[aisle_id] = aisle_counts.get(aisle_id, 0) + 1

    if not aisle_counts:
        # Fallback: no aisle info, use global popularity
        return recommend_tier1(n)

    # Sort aisles by count descending, pick top aisle
    top_aisle_id = max(aisle_counts, key=aisle_counts.get)

    # Get top products for that aisle
    aisle_key = str(int(top_aisle_id))
    aisle_products = aisle_top10.get(aisle_key, [])

    # Filter out current cart items (per D-02; past purchases are allowed)
    recs = [pid for pid in aisle_products if pid not in cart_set]

    # Pad with global_top20 if fewer than n (per D-03)
    if len(recs) < n:
        seen = set(recs) | cart_set
        for pid in global_top20:
            if pid not in seen:
                recs.append(pid)
                seen.add(pid)
            if len(recs) >= n:
                break

    return recs[:n]

# Test with synthetic user history
sample_products = order_products['product_id'].value_counts().head(30).index.tolist()
sample_history = sample_products[:5]   # simulate user who bought top products
sample_cart = [sample_products[0]]     # one item currently in cart

t2_recs = recommend_tier2(sample_history, cart_items=sample_cart, n=5)
print(f'Tier 2 recs (aisle affinity): {t2_recs}')
print(f'Cart item {sample_cart[0]} filtered out: {sample_cart[0] not in t2_recs}')
assert len(t2_recs) == 5
assert sample_cart[0] not in t2_recs, 'Cart items should be filtered per D-02'
print('recommend_tier2: assertions passed')

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Load the Time-Aware LSTM model trained in Phase 5
model = keras.models.load_model(f'{MODELS_DIR}/lstm_time_model.keras')
print(f'Loaded LSTM+Time model: {model.name}')
print(f'Input names: {[inp.name for inp in model.inputs]}')

In [ ]:
def recommend_tier3(user_sequence, hour=12, dow=3, days_gap=7.0, cart_items=None, n=5):
    """Tier 3: LSTM+Time model recommendations for users with 3+ orders.

    Args:
        user_sequence: list of product_ids from user's order history
        hour: hour of day (0-23) for time context
        dow: day of week (0-6) for time context
        days_gap: days since last order (raw; normalized by /30 internally)
        cart_items: list of product_ids currently in cart (to exclude)
        n: number of recommendations to return (default 5)

    Returns: list of exactly n product_id integers
    """
    if cart_items is None:
        cart_items = []
    cart_set = set(cart_items)

    # Map product_ids to vocab indices
    seq_indices = []
    for pid in user_sequence:
        idx = vocab.get(str(pid), 0)  # 0 = padding for unknown products
        seq_indices.append(idx)

    # Pad/truncate to length 99 (SEQ_LEN from Phase 5)
    SEQ_LEN = 99
    if len(seq_indices) > SEQ_LEN:
        seq_indices = seq_indices[-SEQ_LEN:]      # keep most recent
    else:
        seq_indices = [0] * (SEQ_LEN - len(seq_indices)) + seq_indices  # left-pad

    # Prepare model inputs
    seq_input       = np.array([seq_indices], dtype=np.int32)           # (1, 99)
    hour_input      = np.array([hour], dtype=np.int32)                  # (1,)
    dow_input       = np.array([dow], dtype=np.int32)                   # (1,)
    days_gap_input  = np.array([[days_gap / 30.0]], dtype=np.float32)   # (1, 1) normalized

    # Run inference
    preds = model.predict({
        'seq_input': seq_input,
        'hour_input': hour_input,
        'dow_input': dow_input,
        'days_gap_input': days_gap_input
    }, verbose=0)  # (1, 5001)

    probs = preds[0]  # (5001,)

    # Get top indices, skip padding (index 0) and cart items
    sorted_indices = np.argsort(probs)[::-1]

    recs = []
    for idx in sorted_indices:
        if idx == 0:
            continue
        product_id = index_to_product.get(int(idx))
        if product_id is None:
            continue
        if product_id in cart_set:
            continue
        recs.append(product_id)
        if len(recs) >= n:
            break

    # Pad with global_top20 if fewer than n (per D-03)
    if len(recs) < n:
        seen = set(recs) | cart_set
        for pid in global_top20:
            if pid not in seen:
                recs.append(pid)
                seen.add(pid)
            if len(recs) >= n:
                break

    return recs[:n]

# Test with a real user's sequence from the dataset
sample_user_orders = orders[orders['user_id'] == orders['user_id'].iloc[0]].sort_values('order_number')
sample_order_ids = sample_user_orders['order_id'].tolist()
sample_user_products = order_products[order_products['order_id'].isin(sample_order_ids)]['product_id'].tolist()

t3_recs = recommend_tier3(sample_user_products, hour=14, dow=2, days_gap=7.0, n=5)
print(f'Tier 3 recs (LSTM+Time): {t3_recs}')
assert len(t3_recs) == 5
print('recommend_tier3: assertions passed')

In [ ]:
def get_recommendations(order_count, user_order_history=None, cart_items=None,
                        hour=12, dow=3, days_gap=7.0, aisle_id=None, n=5):
    """Unified recommendation router based on user tier.

    Args:
        order_count: number of past orders (determines tier)
        user_order_history: list of product_ids from past orders (Tier 2 and 3)
        cart_items: list of product_ids in current cart (filter out per D-02)
        hour: hour of day 0-23 (Tier 3 time context)
        dow: day of week 0-6 (Tier 3 time context)
        days_gap: days since last order (Tier 3 time context)
        aisle_id: optional aisle_id for Tier 2 when no product history (demo users)
        n: number of recommendations (default 5, per D-03)

    Returns: dict with keys 'tier' and 'recommendations' (list of n product_ids)
    """
    tier = get_user_tier(order_count)

    if tier == 1:
        recs = recommend_tier1(n)
    elif tier == 2:
        history = user_order_history or []
        if not history and aisle_id is not None:
            # Demo users: build synthetic history from the specified aisle
            history = aisle_top10.get(str(int(aisle_id)), [])[:3]
        recs = recommend_tier2(history, cart_items, n)
    elif tier == 3:
        recs = recommend_tier3(user_order_history or [], hour, dow, days_gap, cart_items, n)
    else:
        recs = recommend_tier1(n)  # safety fallback

    return {'tier': tier, 'recommendations': recs}

# Test all tiers
print('=' * 50)
print('Unified Router Tests')
print('=' * 50)

# Tier 1: new user
r1 = get_recommendations(order_count=0, n=5)
print(f"Tier 1 (0 orders): tier={r1['tier']}, recs={r1['recommendations']}")
assert r1['tier'] == 1
assert len(r1['recommendations']) == 5

# Tier 2: minimal history user
r2 = get_recommendations(order_count=2, user_order_history=sample_history,
                         cart_items=[sample_history[0]], n=5)
print(f"Tier 2 (2 orders): tier={r2['tier']}, recs={r2['recommendations']}")
assert r2['tier'] == 2
assert len(r2['recommendations']) == 5
assert sample_history[0] not in r2['recommendations'], 'Cart item should be filtered'

# Tier 3: established user
r3 = get_recommendations(order_count=6, user_order_history=sample_user_products,
                         hour=14, dow=2, days_gap=7.0, n=5)
print(f"Tier 3 (6 orders): tier={r3['tier']}, recs={r3['recommendations']}")
assert r3['tier'] == 3
assert len(r3['recommendations']) == 5

print('=' * 50)
print('All router tests passed. Every tier returns exactly 5 recommendations.')
print('=' * 50)

## Cold Start Evaluation

All dataset users have 6+ orders, so no real Tier 1 or Tier 2 users exist in the data. To evaluate cold start quality, we **simulate** cold start by treating test users as if they had 0 or 1–2 orders:
- **Tier 1** evaluation: Recommend globally popular items; check if the user's ground-truth next item is in top-K
- **Tier 2** evaluation: Use only the first few products of each test user's sequence as a simulated "short history"
- **Tier 3** evaluation: Use the full sequence with the LSTM+Time model (establishes the personalization ceiling)

In [ ]:
def hits_at_k(recommendations, true_item, k):
    """Return 1 if true_item is in top-K recommendations, else 0."""
    return int(true_item in recommendations[:k])

# Load test data for ground truth
test_data = np.load(f'{DATA_DIR}/X_test.npz')
X_test = test_data['sequences']   # (1000, 100)
X_test_seq = X_test[:, :-1]       # (1000, 99) context
y_test = X_test[:, -1]            # (1000,) ground truth next item index

# Map ground truth indices back to product_ids
y_test_products = [index_to_product.get(int(idx), -1) for idx in y_test]

N_EVAL = 200  # Evaluate on 200 test users (meaningful metrics, fast)
print(f'Test set shape: {X_test.shape}')
print(f'Evaluating on {N_EVAL} test users')
print(f'Ground truth product IDs (first 5): {y_test_products[:5]}')

In [ ]:
print('=' * 50)
print('Tier 1 Evaluation: Global Popularity')
print('=' * 50)

tier1_recs = recommend_tier1(n=20)  # top 20 for HR@5 and HR@10

hr5_t1 = []
hr10_t1 = []
for i in range(N_EVAL):
    true_pid = y_test_products[i]
    hr5_t1.append(hits_at_k(tier1_recs, true_pid, 5))
    hr10_t1.append(hits_at_k(tier1_recs, true_pid, 10))

tier1_metrics = {
    'HR@5': np.mean(hr5_t1),
    'HR@10': np.mean(hr10_t1),
}
print(f"Tier 1 HR@5:  {tier1_metrics['HR@5']:.4f}")
print(f"Tier 1 HR@10: {tier1_metrics['HR@10']:.4f}")
print('(Expected: low -- same recs for everyone, no personalization)')

In [ ]:
print('=' * 50)
print('Tier 2 Evaluation: Aisle Affinity (simulated 1-2 orders)')
print('=' * 50)

hr5_t2 = []
hr10_t2 = []

for i in range(N_EVAL):
    true_pid = y_test_products[i]

    # Simulate limited history: take first 5 non-zero items (1-2 orders worth)
    seq = X_test_seq[i]
    non_zero = seq[seq > 0]
    if len(non_zero) < 3:
        continue

    limited_history_indices = non_zero[:5].tolist()
    limited_history_pids = [index_to_product.get(int(idx), -1) for idx in limited_history_indices]
    limited_history_pids = [p for p in limited_history_pids if p > 0]

    if not limited_history_pids:
        continue

    recs = recommend_tier2(limited_history_pids, cart_items=[], n=20)
    hr5_t2.append(hits_at_k(recs, true_pid, 5))
    hr10_t2.append(hits_at_k(recs, true_pid, 10))

tier2_metrics = {
    'HR@5': np.mean(hr5_t2) if hr5_t2 else 0.0,
    'HR@10': np.mean(hr10_t2) if hr10_t2 else 0.0,
}
print(f"Tier 2 HR@5:  {tier2_metrics['HR@5']:.4f} (evaluated on {len(hr5_t2)} users)")
print(f"Tier 2 HR@10: {tier2_metrics['HR@10']:.4f}")
print('(Expected: slightly better than Tier 1 -- aisle narrows the search space)')

In [ ]:
print('=' * 50)
print('Tier 3 Evaluation: LSTM+Time Model')
print('=' * 50)

# Generate synthetic time features for test users (consistent seed for reproducibility)
np.random.seed(42)
hour_test   = np.random.randint(0, 24, size=(len(X_test_seq),)).astype(np.int32)
dow_test    = np.random.randint(0, 7, size=(len(X_test_seq),)).astype(np.int32)
days_test   = (np.random.uniform(1, 30, size=(len(X_test_seq),)).astype(np.float32)) / 30.0

# Batch predict all N_EVAL test users
all_preds = model.predict({
    'seq_input': X_test_seq[:N_EVAL],
    'hour_input': hour_test[:N_EVAL],
    'dow_input': dow_test[:N_EVAL],
    'days_gap_input': days_test[:N_EVAL].reshape(-1, 1)
}, batch_size=512, verbose=1)

hr5_t3 = []
hr10_t3 = []

for i in range(N_EVAL):
    probs = all_preds[i]
    sorted_indices = np.argsort(probs)[::-1]
    top_pids = []
    for idx in sorted_indices:
        if idx == 0:
            continue
        pid = index_to_product.get(int(idx))
        if pid is not None:
            top_pids.append(pid)
        if len(top_pids) >= 20:
            break

    true_pid = y_test_products[i]
    hr5_t3.append(hits_at_k(top_pids, true_pid, 5))
    hr10_t3.append(hits_at_k(top_pids, true_pid, 10))

tier3_metrics = {
    'HR@5': np.mean(hr5_t3),
    'HR@10': np.mean(hr10_t3),
}
print(f"Tier 3 HR@5:  {tier3_metrics['HR@5']:.4f}")
print(f"Tier 3 HR@10: {tier3_metrics['HR@10']:.4f}")
print('(Expected: best -- personalized LSTM+Time predictions)')

## Cold Start System — Results Summary

In [ ]:
summary = pd.DataFrame({
    'Tier': ['Tier 1 (Global Popular)', 'Tier 2 (Aisle Affinity)', 'Tier 3 (LSTM+Time)'],
    'User Profile': ['0 orders (new user)', '1-2 orders (minimal)', '3+ orders (established)'],
    'HR@5': [tier1_metrics['HR@5'], tier2_metrics['HR@5'], tier3_metrics['HR@5']],
    'HR@10': [tier1_metrics['HR@10'], tier2_metrics['HR@10'], tier3_metrics['HR@10']],
})

display_summary = summary.copy()
for col in ['HR@5', 'HR@10']:
    display_summary[col] = display_summary[col].map(lambda x: f'{x:.4f}')

print('=' * 70)
print('FreshCart AI -- Cold Start System Evaluation')
print('=' * 70)
print(display_summary.to_string(index=False))
print('=' * 70)

# Verify tier progression: Tier 3 > Tier 1, Tier 2 >= Tier 1
tier1_hr5 = tier1_metrics['HR@5']
tier2_hr5 = tier2_metrics['HR@5']
tier3_hr5 = tier3_metrics['HR@5']

print(f'\nTier progression check:')
print(f'  tier3_hr5 ({tier3_hr5:.4f}) > tier1_hr5 ({tier1_hr5:.4f}): {tier3_hr5 > tier1_hr5}')
print(f'  tier2_hr5 ({tier2_hr5:.4f}) >= tier1_hr5 ({tier1_hr5:.4f}): {tier2_hr5 >= tier1_hr5}')

assert tier3_hr5 > tier1_hr5, f'Tier 3 HR@5 ({tier3_hr5}) should exceed Tier 1 HR@5 ({tier1_hr5})'
assert tier2_hr5 >= tier1_hr5, f'Tier 2 HR@5 ({tier2_hr5}) should be >= Tier 1 HR@5 ({tier1_hr5})'
print('\nTier progression verified: Tier 3 > Tier 1 and Tier 2 >= Tier 1 confirmed.')

print()
print('Key observations:')
print('- Tier 1 provides a reasonable baseline using global popularity')
print('- Tier 2 improves by narrowing to the user\'s preferred aisle')
print('- Tier 3 provides the best personalization using the full LSTM+Time model')
print('- The 3-tier system ensures every user gets recommendations regardless of history')

## Demo User Validation (SC-3, SC-4)

Phase 8 will create 5 hardcoded demo users in `users.json`. Here we validate that the cold start routing logic produces correct results for the two cold start demo profiles:
- **new_user_01** (0 orders) → must route to Tier 1 and return 5 product IDs *(SC-3)*
- **neha_gupta** (2 orders, fresh vegetables aisle affinity) → must route to Tier 2 and return 5 product IDs *(SC-4)*

In [ ]:
# Demo user profiles (from PRD Section 3.2)
# Full users.json is created in Phase 8; here we verify routing logic
demo_users = {
    'new_user_01': {
        'user_id': 5,
        'display_name': 'New User',
        'order_count': 0,
        'cart': [],
        'aisle_history': []
    },
    'neha_gupta': {
        'user_id': 4,
        'display_name': 'Neha Gupta',
        'order_count': 2,
        'cart': [],
        # fresh vegetables aisle_id = 83 in Instacart dataset
        'aisle_history': [83]
    }
}

print('=' * 60)
print('Demo User Validation')
print('=' * 60)

# --- SC-3: new_user_01 (Tier 1) ---
nu = demo_users['new_user_01']
assert get_user_tier(nu['order_count']) == 1, \
    f"new_user_01 should be Tier 1, got Tier {get_user_tier(nu['order_count'])}"

r_nu = get_recommendations(
    order_count=nu['order_count'],
    cart_items=nu['cart'],
    hour=10, dow=3, days_gap=30.0,
    n=5
)
assert r_nu['tier'] == 1, f"new_user_01 tier should be 1, got {r_nu['tier']}"
assert isinstance(r_nu['recommendations'], list)
assert len(r_nu['recommendations']) == 5, \
    f"new_user_01 should get 5 recs, got {len(r_nu['recommendations'])}"
assert all(isinstance(pid, int) for pid in r_nu['recommendations'])

print(f"SC-3 new_user_01: tier={r_nu['tier']}, recs={r_nu['recommendations']}")
print('  -> Tier 1 (global popularity) [PASS]')

# --- SC-4: neha_gupta (Tier 2) ---
ng = demo_users['neha_gupta']
assert get_user_tier(ng['order_count']) == 2, \
    f"neha_gupta should be Tier 2, got Tier {get_user_tier(ng['order_count'])}"

r_ng = get_recommendations(
    order_count=ng['order_count'],
    cart_items=ng['cart'],
    aisle_id=ng['aisle_history'][0] if ng['aisle_history'] else None,
    hour=10, dow=3, days_gap=30.0,
    n=5
)
assert r_ng['tier'] == 2, f"neha_gupta tier should be 2, got {r_ng['tier']}"
assert isinstance(r_ng['recommendations'], list)
assert len(r_ng['recommendations']) == 5, \
    f"neha_gupta should get 5 recs, got {len(r_ng['recommendations'])}"
assert all(isinstance(pid, int) for pid in r_ng['recommendations'])

print(f"SC-4 neha_gupta:  tier={r_ng['tier']}, recs={r_ng['recommendations']}")
print('  -> Tier 2 (aisle affinity, fresh vegetables) [PASS]')

print()
print('=' * 60)
print('Both demo user profiles validated successfully.')
print('  SC-3: new_user_01 -> Tier 1, 5 product IDs returned')
print('  SC-4: neha_gupta  -> Tier 2, 5 product IDs returned')
print('=' * 60)

## Phase 6: Cold Start System — Complete

**What was built:**
1. **Precomputed lookup tables** (4 JSON files in `data/precomputed/`):
   - `global_top20.json` — 20 most popular products globally
   - `hourly_top10.json` — top 10 products per hour of day (24 keys)
   - `dow_top10.json` — top 10 products per day of week (7 keys)
   - `aisle_top10.json` — top 10 products per aisle (134 aisles)

2. **3-tier routing logic:**
   - `get_user_tier(order_count)` — determines tier from order count
   - `recommend_tier1()` — global popularity for brand-new users
   - `recommend_tier2()` — aisle affinity for minimal-history users
   - `recommend_tier3()` — LSTM+Time for established users
   - `get_recommendations()` — unified router returning `{tier, recommendations}`

3. **Design decisions implemented:**
   - D-01: JSON files store only `product_id` arrays (no full product objects)
   - D-02: Tier 2 filters cart items; past purchases allowed
   - D-03: All tiers pad with global_top20 to always return exactly 5 recommendations

4. **Validated demo users:**
   - SC-3: `new_user_01` (0 orders) → Tier 1, returns 5 globally popular product IDs
   - SC-4: `neha_gupta` (2 orders) → Tier 2, returns 5 aisle-affinity product IDs

**Next:** Phase 7 (Full Evaluation) will run the complete 4-system comparison across all metrics.